# Customer Engagement & Product Utilization Analytics for Retention Strategy
### Unified Mentor & European Central Bank Analytics Project
This notebook conducts end-to-end data validation, exploratory data analysis, feature engineering, predictive modeling, and model evaluation for bank customer retention.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score
import joblib

# Load dataset
df = pd.read_csv('3.csv')
print("Dataset Shape:", df.shape)
df.head()

## 1. Exploratory Data Analysis & Business KPIs
Let us examine key churn distributions, product depth indices, and engagement metrics.

In [2]:
# Churn rate by NumOfProducts
prod_churn = df.groupby('NumOfProducts')['Exited'].agg(['count', 'mean'])
print("Churn by Product Count:\n", prod_churn)

# Churn by Active Member
act_churn = df.groupby('IsActiveMember')['Exited'].agg(['count', 'mean'])
print("\nChurn by Activity Status:\n", act_churn)

# Churn by Geography
geo_churn = df.groupby('Geography')['Exited'].agg(['count', 'mean'])
print("\nChurn by Geography:\n", geo_churn)

## 2. Feature Engineering & Preprocessing Pipeline

In [3]:
df_clean = df.drop(columns=['Year', 'CustomerId', 'Surname'])

# Feature engineering
df_clean['BalanceSalaryRatio'] = df_clean['Balance'] / (df_clean['EstimatedSalary'] + 1)
df_clean['TenureAgeRatio'] = df_clean['Tenure'] / (df_clean['Age'] + 1)
df_clean['CreditScoreAgeRatio'] = df_clean['CreditScore'] / (df_clean['Age'] + 1)
df_clean['IsZeroBalance'] = (df_clean['Balance'] == 0).astype(int)
df_clean['EngagementScore'] = df_clean['IsActiveMember'] * 2 + df_clean['HasCrCard'] + (df_clean['NumOfProducts'] >= 2).astype(int)

X = df_clean.drop('Exited', axis=1)
y = df_clean['Exited']

categorical_cols = ['Geography', 'Gender']
numerical_cols = [c for c in X.columns if c not in categorical_cols]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
    ]
)

## 3. Model Training & Benchmarking

In [4]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=4, random_state=42)
}

for name, m in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('clf', m)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)[:, 1]
    print(f"=== {name} ===")
    print(f"Accuracy: {accuracy_score(y_test, preds):.4f} | ROC-AUC: {roc_auc_score(y_test, probs):.4f}")
    print(classification_report(y_test, preds))

## 4. Production Model Serialization

In [5]:
final_pipe = Pipeline([('preprocessor', preprocessor), ('clf', GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=4, random_state=42))])
final_pipe.fit(X, y)
joblib.dump(final_pipe, 'model.pkl')
print("Exported model.pkl successfully!")